# GPU Budget Negotiation Arena — Standalone Eval Notebook

**Open in Colab:** <https://colab.research.google.com/github/abhinavgautam01/GPU_Budget_Negotiation_Arena/blob/main/notebooks/eval.ipynb>

Reproduce the headline trained-LLM-vs-baselines numbers in **~5 minutes on a free T4**, without re-running any training.

What this notebook does:

1. Clones the repo and installs runtime deps (no Unsloth/TRL needed — eval-only stack).
2. Points `--model-path` at your SFT LoRA (default: `artifacts/sft-checkpoint` after you copy it from Google Drive, or any local path — no Hub upload required).
3. Runs the SFT'd model and the *untrained* `Llama-3.2-3B-Instruct` head-to-head against four scripted baselines on `single_trade`, `market_round`, and `coalition_market` for 5 seeds each.
4. Renders the headline plot (`plots/trained_llm_vs_baselines.svg`) and prints the comparison table.

If `--include-base-model` is too expensive on your runtime, drop the flag and you'll still see the trained model vs the four scripted baselines.

> **Why a separate notebook:** the main `training/GPU_Budget_Negotiation_Arena_Colab.ipynb` is the full pipeline (SFT → GRPO → eval). This one is the *evaluation-only* surface a judge can run in a coffee break to verify our reward-improvement numbers.

## 1. Setup

In [ ]:
%%bash
set -e
if [ ! -d GPU_Budget_Negotiation_Arena ]; then
    git clone --depth 1 https://github.com/abhinavgautam01/GPU_Budget_Negotiation_Arena.git
fi
cd GPU_Budget_Negotiation_Arena
pip install -q -r requirements.txt
pip install -q peft==0.13.2 bitsandbytes==0.44.1
echo OK

In [ ]:
%cd GPU_Budget_Negotiation_Arena
import os, sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
print('cwd:', pathlib.Path.cwd())

## 2. Configure the adapter you want to evaluate

Default: `artifacts/sft-checkpoint` (local, or copy from Drive). You can also set a Hugging Face model id here if you host the adapter there.

In [ ]:
BASE_MODEL = 'unsloth/Llama-3.2-3B-Instruct'
ADAPTER    = 'artifacts/sft-checkpoint'  # e.g. copy from Drive: .../gpu_budget_negotiation_arena/artifacts/sft-checkpoint
POLICY_NAME = 'trained_llm_sft'
SEEDS = '5'        # 5 seeds × 3 tasks = 15 trained episodes (+ 15 base if --include-base-model)
INCLUDE_BASE = True

## 3. Run trained-LLM-vs-baselines evaluation

In [ ]:
import subprocess, shlex
cmd = [
    'python3', 'scripts/evaluate_trained_llm.py',
    '--base-model', BASE_MODEL,
    '--model-path', ADAPTER,
    '--policy-name', POLICY_NAME,
    '--seeds', SEEDS,
    '--output', 'artifacts/trained_llm_eval.json',
]
if INCLUDE_BASE:
    cmd.append('--include-base-model')
print('+', ' '.join(shlex.quote(c) for c in cmd))
subprocess.run(cmd, check=True)
print(open('artifacts/trained_llm_eval.json').read()[:600])

## 4. Render the comparison plot

In [ ]:
subprocess.run([
    'python3', 'scripts/plot_trained_vs_baselines.py',
    '--eval', 'artifacts/trained_llm_eval.json',
    '--baselines', 'artifacts/baseline_eval.json',
    '--plot', 'plots/trained_llm_vs_baselines.svg',
    '--summary', 'artifacts/trained_llm_summary.json',
], check=True)
from IPython.display import SVG, display
display(SVG('plots/trained_llm_vs_baselines.svg'))

## 5. Print the headline table

In [ ]:
import json
summary = json.load(open('artifacts/trained_llm_summary.json'))
rows = summary.get('rows', [])
headers = ['policy', 'single_trade_mean', 'market_round_mean', 'coalition_market_mean', 'overall_mean']
print(' | '.join(h.ljust(22) for h in headers))
print('-' * (22 * len(headers) + 3 * (len(headers) - 1)))
for r in rows:
    cells = []
    for h in headers:
        v = r.get(h, '')
        if isinstance(v, float):
            cells.append(f'{v:+.4f}'.ljust(22))
        else:
            cells.append(str(v).ljust(22))
    print(' | '.join(cells))

**Expected outcome (matches `artifacts/trained_llm_summary.json` shipped in the repo):**

- Sign flips on `market_round` and `coalition_market` for the SFT'd Llama vs the same base Llama (untrained), with the trained policy ranking 3rd of 8 evaluated policies on overall mean reward.
- Always-accept beats us by `~0.01` on the structural scalar — see `README.md → 'Honest framing'` for why we still come out ahead under the judge layer and on coalition reliability under capacity shocks.
- Parse-failure rate ~50% at the SFT checkpoint; the GRPO stage continues to drive that down (see `plots/grpo_reward_curve.svg`).